In [ ]:
"""
Text Preprocessing Pipeline
- Remove special characters
- Remove duplicates
- Remove PII (emails, phones, SSN, names)
- Remove PCI (credit card numbers, CVV, etc.)
- Remove abusive/frustrating/hateful content (keyword-based)
"""

import pandas as pd
import numpy as np
import re
from typing import List, Set, Dict, Tuple, Optional
from dataclasses import dataclass, field
from collections import Counter
import hashlib
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")


@dataclass
class FilterStats:
  """Track filtering statistics."""
  initial_count: int = 0
  after_special_chars: int = 0
  after_duplicates: int = 0
  after_pii: int = 0
  after_pci: int = 0
  after_abusive: int = 0
  final_count: int = 0

  removed_special_chars: int = 0
  removed_duplicates: int = 0
  removed_pii: int = 0
  removed_pci: int = 0
  removed_abusive: int = 0

  def print_summary(self):
      print(f"\n{'='*60}")
      print("PREPROCESSING SUMMARY")
      print(f"{'='*60}")
      print(f"Initial records:           {self.initial_count:,}")
      print(f"\nRemoved at each step:")
      print(f"  Special characters:      {self.removed_special_chars:,}")
      print(f"  Duplicates:              {self.removed_duplicates:,}")
      print(f"  PII (personal info):     {self.removed_pii:,}")
      print(f"  PCI (payment data):      {self.removed_pci:,}")
      print(f"  Abusive/Hateful:         {self.removed_abusive:,}")
      print(f"\nFinal records:             {self.final_count:,}")
      print(f"Retention rate:            {100*self.final_count/self.initial_count:.1f}%")
      print(f"{'='*60}")


# =============================================================================
# 1. SPECIAL CHARACTERS FILTER
# =============================================================================

class SpecialCharacterFilter:
  """Remove entries with excessive special characters."""

  def __init__(
      self,
      max_special_ratio: float = 0.3,      # Max 30% special chars
      allowed_special: str = ".,!?'-$%",   # Allowed special chars
      min_alpha_ratio: float = 0.5,        # Min 50% alphabetic
      remove_urls: bool = True,
      remove_html: bool = True
  ):
      self.max_special_ratio = max_special_ratio
      self.allowed_special = set(allowed_special)
      self.min_alpha_ratio = min_alpha_ratio
      self.remove_urls = remove_urls
      self.remove_html = remove_html

      # Patterns
      self.url_pattern = re.compile(r'https?://\S+|www\.\S+')
      self.html_pattern = re.compile(r'<[^>]+>')
      self.repeated_chars = re.compile(r'(.)\1{4,}')  # 5+ repeated chars
      self.excessive_punctuation = re.compile(r'[!?.,]{3,}')

  def is_valid(self, text: str) -> Tuple[bool, str]:
      """Check if text passes special character filters."""

      if not text or not isinstance(text, str):
          return False, "empty"

      text = text.strip()

      if len(text) < 2:
          return False, "too_short"

      # Check for URLs
      if self.remove_urls and self.url_pattern.search(text):
          return False, "contains_url"

      # Check for HTML
      if self.remove_html and self.html_pattern.search(text):
          return False, "contains_html"

      # Check for repeated characters (aaaaaaa, !!!!!!!)
      if self.repeated_chars.search(text):
          return False, "repeated_chars"

      # Calculate character ratios
      text_no_space = text.replace(" ", "")
      if len(text_no_space) == 0:
          return False, "only_whitespace"

      alpha_count = sum(c.isalpha() for c in text_no_space)
      digit_count = sum(c.isdigit() for c in text_no_space)
      special_count = sum(
          not c.isalnum() and c not in self.allowed_special
          for c in text_no_space
      )

      alpha_ratio = alpha_count / len(text_no_space)
      special_ratio = special_count / len(text_no_space)

      if alpha_ratio < self.min_alpha_ratio:
          return False, "low_alpha_ratio"

      if special_ratio > self.max_special_ratio:
          return False, "high_special_ratio"

      return True, "valid"

  def filter(self, df: pd.DataFrame, text_column: str = "text") -> pd.DataFrame:
      """Filter DataFrame."""
      mask = df[text_column].apply(lambda x: self.is_valid(x)[0])
      return df[mask].copy()


# =============================================================================
# 2. DUPLICATE FILTER
# =============================================================================

class DuplicateFilter:
  """Remove duplicate and near-duplicate entries."""

  def __init__(
      self,
      case_sensitive: bool = False,
      normalize_whitespace: bool = True,
      remove_punctuation_for_comparison: bool = True
  ):
      self.case_sensitive = case_sensitive
      self.normalize_whitespace = normalize_whitespace
      self.remove_punctuation = remove_punctuation_for_comparison

  def normalize(self, text: str) -> str:
      """Normalize text for comparison."""
      if not isinstance(text, str):
          return ""

      normalized = text

      if not self.case_sensitive:
          normalized = normalized.lower()

      if self.normalize_whitespace:
          normalized = re.sub(r'\s+', ' ', normalized).strip()

      if self.remove_punctuation:
          normalized = re.sub(r'[^\w\s]', '', normalized)

      return normalized

  def get_hash(self, text: str) -> str:
      """Get hash of normalized text."""
      return hashlib.md5(self.normalize(text).encode()).hexdigest()

  def filter(self, df: pd.DataFrame, text_column: str = "text") -> pd.DataFrame:
      """Remove duplicates based on normalized text."""
      df = df.copy()
      df["_normalized_hash"] = df[text_column].apply(self.get_hash)
      df_deduped = df.drop_duplicates(subset=["_normalized_hash"], keep="first")
      df_deduped = df_deduped.drop(columns=["_normalized_hash"])
      return df_deduped


# =============================================================================
# 3. PII FILTER (Personally Identifiable Information)
# =============================================================================

class PIIFilter:
  """Remove entries containing PII."""

  def __init__(self):
      # Email pattern
      self.email_pattern = re.compile(
          r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
      )

      # Phone patterns (various formats)
      self.phone_patterns = [
          re.compile(r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b'),  # 123-456-7890
          re.compile(r'\(\d{3}\)\s*\d{3}[-.\s]?\d{4}'),       # (123) 456-7890
          re.compile(r'\+\d{1,3}[-.\s]?\d{3,14}'),            # +1 234 567 8900
          re.compile(r'\b\d{10,11}\b'),                        # 1234567890
      ]

      # SSN pattern
      self.ssn_pattern = re.compile(
          r'\b\d{3}[-\s]?\d{2}[-\s]?\d{4}\b'
      )

      # Date of birth patterns
      self.dob_patterns = [
          re.compile(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b'),  # MM/DD/YYYY
          re.compile(r'\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b'),    # YYYY-MM-DD
      ]

      # Address patterns (basic)
      self.address_patterns = [
          re.compile(r'\b\d+\s+\w+\s+(street|st|avenue|ave|road|rd|boulevard|blvd|drive|dr|lane|ln|way|court|ct)\b', re.I),
          re.compile(r'\b(apt|apartment|suite|ste|unit)\s*#?\s*\d+\b', re.I),
          re.compile(r'\b\d{5}(-\d{4})?\b'),  # ZIP code
      ]

      # Name patterns (common name indicators)
      self.name_patterns = [
          re.compile(r'\b(my name is|i am|this is)\s+([A-Z][a-z]+\s+[A-Z][a-z]+)', re.I),
          re.compile(r'\b(mr|mrs|ms|dr|miss)\.\s*[A-Z][a-z]+', re.I),
      ]

      # IP Address
      self.ip_pattern = re.compile(
          r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'
      )

  def contains_pii(self, text: str) -> Tuple[bool, List[str]]:
      """Check if text contains PII."""

      if not isinstance(text, str):
          return False, []

      found_pii = []

      # Check email
      if self.email_pattern.search(text):
          found_pii.append("email")

      # Check phone
      for pattern in self.phone_patterns:
          if pattern.search(text):
              found_pii.append("phone")
              break

      # Check SSN
      if self.ssn_pattern.search(text):
          # Validate it's likely an SSN (not a date or other number)
          match = self.ssn_pattern.search(text)
          if match:
              ssn_text = re.sub(r'[-\s]', '', match.group())
              # Basic SSN validation: not all same digit, valid area number
              if len(set(ssn_text)) > 1 and ssn_text[:3] not in ['000', '666']:
                  found_pii.append("ssn")

      # Check addresses
      for pattern in self.address_patterns:
          if pattern.search(text):
              found_pii.append("address")
              break

      # Check names
      for pattern in self.name_patterns:
          if pattern.search(text):
              found_pii.append("name")
              break

      # Check IP
      if self.ip_pattern.search(text):
          found_pii.append("ip_address")

      return len(found_pii) > 0, found_pii

  def filter(self, df: pd.DataFrame, text_column: str = "text") -> pd.DataFrame:
      """Remove entries with PII."""
      mask = df[text_column].apply(lambda x: not self.contains_pii(x)[0])
      return df[mask].copy()


# =============================================================================
# 4. PCI FILTER (Payment Card Industry Data)
# =============================================================================

class PCIFilter:
  """Remove entries containing PCI data (credit card info)."""

  def __init__(self):
      # Credit card patterns (various formats)
      self.cc_patterns = [
          # Visa: starts with 4
          re.compile(r'\b4\d{3}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b'),
          # Mastercard: starts with 51-55 or 2221-2720
          re.compile(r'\b5[1-5]\d{2}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b'),
          re.compile(r'\b2[2-7]\d{2}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b'),
          # Amex: starts with 34 or 37
          re.compile(r'\b3[47]\d{2}[-\s]?\d{6}[-\s]?\d{5}\b'),
          # Discover: starts with 6011, 65, or 644-649
          re.compile(r'\b6(?:011|5\d{2}|4[4-9]\d)[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b'),
          # Generic 16-digit
          re.compile(r'\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b'),
          # Generic 15-digit (Amex)
          re.compile(r'\b\d{4}[-\s]?\d{6}[-\s]?\d{5}\b'),
      ]

      # CVV patterns
      self.cvv_patterns = [
          re.compile(r'\b(cvv|cvc|cv2|cid|security code)[:\s]*\d{3,4}\b', re.I),
          re.compile(r'\b\d{3,4}\s*(cvv|cvc|cv2|cid|security code)\b', re.I),
      ]

      # Expiry patterns
      self.expiry_patterns = [
          re.compile(r'\b(exp|expiry|expiration|valid)[:\s]*(thru|through|until)?[:\s]*\d{1,2}[/-]\d{2,4}\b', re.I),
          re.compile(r'\b\d{1,2}[/-]\d{2,4}\s*(exp|expiry|expiration)\b', re.I),
          re.compile(r'\bMM[/-]YY\b', re.I),
      ]

      # Bank account patterns
      self.bank_patterns = [
          re.compile(r'\b(account|acct)[:\s#]*\d{8,17}\b', re.I),
          re.compile(r'\b(routing)[:\s#]*\d{9}\b', re.I),
          re.compile(r'\b(iban)[:\s]*[A-Z]{2}\d{2}[A-Z0-9]{4,30}\b', re.I),
      ]

      # PIN patterns
      self.pin_patterns = [
          re.compile(r'\b(pin|passcode)[:\s]*\d{4,6}\b', re.I),
      ]

      # Card-related keywords
      self.card_keywords = [
          "credit card number",
          "debit card number",
          "card number is",
          "my card is",
          "full card number",
      ]

  def luhn_checksum(self, card_number: str) -> bool:
      """Validate card number using Luhn algorithm."""
      digits = re.sub(r'\D', '', card_number)
      if len(digits) < 13 or len(digits) > 19:
          return False

      total = 0
      for i, digit in enumerate(reversed(digits)):
          d = int(digit)
          if i % 2 == 1:
              d *= 2
              if d > 9:
                  d -= 9
          total += d

      return total % 10 == 0

  def contains_pci(self, text: str) -> Tuple[bool, List[str]]:
      """Check if text contains PCI data."""

      if not isinstance(text, str):
          return False, []

      found_pci = []

      # Check credit card numbers
      for pattern in self.cc_patterns:
          matches = pattern.findall(text)
          for match in matches:
              # Validate with Luhn algorithm
              if self.luhn_checksum(match):
                  found_pci.append("credit_card")
                  break
          if "credit_card" in found_pci:
              break

      # Check CVV
      for pattern in self.cvv_patterns:
          if pattern.search(text):
              found_pci.append("cvv")
              break

      # Check expiry
      for pattern in self.expiry_patterns:
          if pattern.search(text):
              found_pci.append("expiry_date")
              break

      # Check bank accounts
      for pattern in self.bank_patterns:
          if pattern.search(text):
              found_pci.append("bank_account")
              break

      # Check PIN
      for pattern in self.pin_patterns:
          if pattern.search(text):
              found_pci.append("pin")
              break

      # Check keywords
      text_lower = text.lower()
      for keyword in self.card_keywords:
          if keyword in text_lower:
              found_pci.append("card_keyword")
              break

      return len(found_pci) > 0, found_pci

  def filter(self, df: pd.DataFrame, text_column: str = "text") -> pd.DataFrame:
      """Remove entries with PCI data."""
      mask = df[text_column].apply(lambda x: not self.contains_pci(x)[0])
      return df[mask].copy()


# =============================================================================
# 5. ABUSIVE/HATEFUL/FRUSTRATION KEYWORD FILTER
# =============================================================================

class AbusiveContentFilter:
  """Remove abusive, hateful, and frustrating content using keywords."""

  def __init__(self, custom_keywords: Optional[Dict[str, List[str]]] = None):
      # Default keyword lists
      self.keywords = {
          # Profanity
          "profanity": [
              "fuck", "fucking", "fucked", "fucker", "fck", "f*ck", "f**k",
              "shit", "shitting", "bullshit", "sh*t", "sh1t",
              "ass", "asshole", "a**hole", "a$$",
              "bitch", "b*tch", "b1tch",
              "damn", "damned", "dammit",
              "crap", "crappy",
              "bastard", "b@stard",
              "piss", "pissed",
              "hell", "bloody hell",
              "wtf", "stfu", "lmao", "lmfao",
          ],

          # Insults and harassment
          "insults": [
              "idiot", "idiots", "idiotic",
              "stupid", "stupidity",
              "moron", "moronic", "morons",
              "dumb", "dumbass",
              "retard", "retarded",
              "loser", "losers",
              "pathetic",
              "incompetent", "incompetence",
              "useless",
              "worthless",
              "trash",
              "garbage",
              "disgrace",
              "fool", "foolish", "fools",
              "jerk", "jerks",
              "creep", "creepy",
              "scum", "scumbag",
          ],

          # Hate speech
          "hate_speech": [
              # Racial slurs (abbreviated/masked for safety)
              "n*gger", "n1gger", "nigga",
              "sp*c", "sp1c",
              "ch*nk",
              "g**k",
              "w*tback",
              "k*ke",
              "towelhead",
              "sandnigger",
              # Homophobic
              "f*ggot", "f@ggot", "faggot", "fag",
              "dyke",
              "homo",
              "queer",  # context dependent, but flagging
              # Sexist
              "slut", "whore", "hoe",
              "bimbo",
              "feminazi",
              # Religious
              "infidel",
              # General hate
              "subhuman",
              "vermin",
          ],

          # Threats and violence
          "threats": [
              "kill you", "kill myself", "kill him", "kill her", "kill them",
              "gonna kill", "going to kill", "will kill",
              "murder", "murdered",
              "die", "hope you die", "go die",
              "shoot you", "shoot up",
              "bomb", "bombing",
              "attack", "attacking",
              "hurt you", "hurt him", "hurt her",
              "beat you", "beat up",
              "destroy you",
              "burn down", "burn it",
              "stab",
              "punch", "punching",
              "slap",
              "rape",
              "assault",
              "sue you", "lawsuit",
              "revenge",
          ],

          # Frustration and complaints
          "frustration": [
              "frustrated", "frustrating", "frustration",
              "annoyed", "annoying", "annoyance",
              "angry", "anger",
              "hate this", "hate it", "hate you", "i hate",
              "worst", "the worst",
              "terrible", "terribly",
              "horrible", "horribly",
              "awful", "awfully",
              "sucks", "sucked", "suck",
              "broken", "doesn't work", "not working", "won't work",
              "useless", "pointless",
              "waste of time", "wasting my time",
              "ridiculous", "absurd",
              "unacceptable",
              "disappointed", "disappointing", "disappointment",
              "fed up",
              "sick of", "tired of",
              "can't stand",
              "disgusted", "disgusting",
              "outraged", "outrageous",
              "unbelievable",
              "so slow", "too slow", "taking forever",
              "never works", "always broken",
              "piece of junk", "piece of crap",
              "rip off", "ripoff", "scam",
              "ugh", "argh", "grr",
          ],

          # Self-harm (for safety)
          "self_harm": [
              "suicide", "suicidal",
              "kill myself",
              "end my life",
              "want to die",
              "self harm", "self-harm",
              "cut myself", "cutting myself",
              "overdose",
          ],

          # Dangerous/illegal
          "dangerous": [
              "hack", "hacking", "hacker",
              "steal", "stealing", "stolen",
              "fraud", "fraudulent",
              "scam", "scamming",
              "illegal",
              "launder", "laundering",
              "counterfeit",
              "fake id", "fake identity",
              "drug", "drugs",
              "weapon", "weapons", "gun", "guns",
          ],
      }

      # Merge custom keywords
      if custom_keywords:
          for category, words in custom_keywords.items():
              if category in self.keywords:
                  self.keywords[category].extend(words)
              else:
                  self.keywords[category] = words

      # Compile regex patterns for each keyword
      self.patterns = {}
      for category, words in self.keywords.items():
          # Create pattern that matches whole words (case insensitive)
          pattern_str = r'\b(' + '|'.join(re.escape(word) for word in words) + r')\b'
          self.patterns[category] = re.compile(pattern_str, re.IGNORECASE)

  def contains_abusive(self, text: str) -> Tuple[bool, List[str], List[str]]:
      """
      Check if text contains abusive content.
      
      Returns:
          (is_abusive, categories_found, matched_keywords)
      """
      if not isinstance(text, str):
          return False, [], []

      categories_found = []
      matched_keywords = []

      for category, pattern in self.patterns.items():
          matches = pattern.findall(text)
          if matches:
              categories_found.append(category)
              matched_keywords.extend(matches)

      return len(categories_found) > 0, categories_found, matched_keywords

  def filter(
      self, 
      df: pd.DataFrame, 
      text_column: str = "text",
      return_removed: bool = False
  ) -> pd.DataFrame:
      """Remove abusive entries."""

      results = df[text_column].apply(lambda x: self.contains_abusive(x))
      mask = ~results.apply(lambda x: x[0])  # Keep entries where is_abusive is False

      if return_removed:
          removed_df = df[~mask].copy()
          removed_df["abuse_categories"] = results[~mask].apply(lambda x: x[1])
          removed_df["matched_keywords"] = results[~mask].apply(lambda x: x[2])
          return df[mask].copy(), removed_df

      return df[mask].copy()


# =============================================================================
# 6. MAIN PREPROCESSING PIPELINE
# =============================================================================

class TextPreprocessor:
  """Complete text preprocessing pipeline."""

  def __init__(
      self,
      remove_special_chars: bool = True,
      remove_duplicates: bool = True,
      remove_pii: bool = True,
      remove_pci: bool = True,
      remove_abusive: bool = True,
      custom_abusive_keywords: Optional[Dict[str, List[str]]] = None
  ):
      self.stats = FilterStats()

      # Initialize filters
      self.special_char_filter = SpecialCharacterFilter() if remove_special_chars else None
      self.duplicate_filter = DuplicateFilter() if remove_duplicates else None
      self.pii_filter = PIIFilter() if remove_pii else None
      self.pci_filter = PCIFilter() if remove_pci else None
      self.abusive_filter = AbusiveContentFilter(custom_abusive_keywords) if remove_abusive else None

  def process(
      self,
      df: pd.DataFrame,
      text_column: str = "text",
      verbose: bool = True
  ) -> pd.DataFrame:
      """Run complete preprocessing pipeline."""

      self.stats.initial_count = len(df)
      current_df = df.copy()

      if verbose:
          print(f"Starting preprocessing: {len(current_df):,} records")

      # Step 1: Special characters
      if self.special_char_filter:
          prev_count = len(current_df)
          current_df = self.special_char_filter.filter(current_df, text_column)
          self.stats.removed_special_chars = prev_count - len(current_df)
          self.stats.after_special_chars = len(current_df)
          if verbose:
              print(f"After special char filter: {len(current_df):,} (-{self.stats.removed_special_chars:,})")

      # Step 2: Duplicates
      if self.duplicate_filter:
          prev_count = len(current_df)
          current_df = self.duplicate_filter.filter(current_df, text_column)
          self.stats.removed_duplicates = prev_count - len(current_df)
          self.stats.after_duplicates = len(current_df)
          if verbose:
              print(f"After duplicate filter:    {len(current_df):,} (-{self.stats.removed_duplicates:,})")

      # Step 3: PII
      if self.pii_filter:
          prev_count = len(current_df)
          current_df = self.pii_filter.filter(current_df, text_column)
          self.stats.removed_pii = prev_count - len(current_df)
          self.stats.after_pii = len(current_df)
          if verbose:
              print(f"After PII filter:          {len(current_df):,} (-{self.stats.removed_pii:,})")

      # Step 4: PCI
      if self.pci_filter:
          prev_count = len(current_df)
          current_df = self.pci_filter.filter(current_df, text_column)
          self.stats.removed_pci = prev_count - len(current_df)
          self.stats.after_pci = len(current_df)
          if verbose:
              print(f"After PCI filter:          {len(current_df):,} (-{self.stats.removed_pci:,})")

      # Step 5: Abusive content
      if self.abusive_filter:
          prev_count = len(current_df)
          current_df = self.abusive_filter.filter(current_df, text_column)
          self.stats.removed_abusive = prev_count - len(current_df)
          self.stats.after_abusive = len(current_df)
          if verbose:
              print(f"After abusive filter:      {len(current_df):,} (-{self.stats.removed_abusive:,})")

      self.stats.final_count = len(current_df)

      if verbose:
          self.stats.print_summary()

      return current_df.reset_index(drop=True)


# =============================================================================
# 7. SIMPLE USAGE FUNCTION
# =============================================================================

def preprocess_text_data(
  input_csv: str,
  output_csv: str,
  text_column: str = "text",
  custom_keywords: Optional[Dict[str, List[str]]] = None
) -> pd.DataFrame:
  """
  Simple function to preprocess text data.
  
  Args:
      input_csv: Input CSV path
      output_csv: Output CSV path
      text_column: Column name containing text
      custom_keywords: Additional keywords to filter
  
  Returns:
      Cleaned DataFrame
  """
  # Load
  print(f"Loading {input_csv}...")
  df = pd.read_csv(input_csv)

  # Process
  preprocessor = TextPreprocessor(
      remove_special_chars=True,
      remove_duplicates=True,
      remove_pii=True,
      remove_pci=True,
      remove_abusive=True,
      custom_abusive_keywords=custom_keywords
  )

  clean_df = preprocessor.process(df, text_column)

  # Save
  clean_df.to_csv(output_csv, index=False)
  print(f"\nSaved to: {output_csv}")

  return clean_df


# =============================================================================
# 8. MAIN
# =============================================================================

if __name__ == "__main__":
  # Create test data
  test_data = [
      # Clean queries
      "what is my account balance",
      "transfer money to savings",
      "pay my credit card bill",
      "check recent transactions",

      # Special characters
      "!!!@@@###$$$",
      "asdfghjkl;;;;;",
      "check <script>alert('xss')</script>",

      # Duplicates
      "what is my balance",
      "What Is My Balance",  # Case variant
      "what  is  my  balance",  # Whitespace variant

      # PII
      "my email is john@example.com",
      "call me at 555-123-4567",
      "my ssn is 123-45-6789",
      "I live at 123 Main Street",

      # PCI
      "my card is 4111111111111111",
      "cvv 123",
      "card number 4532-1234-5678-9012",

      # Frustration
      "this is so frustrating",
      "why is this app so slow",
      "worst service ever",
      "I hate this bank",

      # Abuse
      "you people are idiots",
      "stupid morons",
      "go to hell",

      # Hate speech
      "you f***ing idiot",

      # Threats
      "I'll sue you all",
  ]

  # Create DataFrame
  df = pd.DataFrame({"text": test_data})
  df.to_csv("/tmp/test_input.csv", index=False)

  # Process
  clean_df = preprocess_text_data(
      input_csv="/tmp/test_input.csv",
      output_csv="/tmp/test_clean.csv",
      text_column="text"
  )

  print("\n" + "="*60)
  print("CLEAN QUERIES")
  print("="*60)
  for text in clean_df["text"].tolist():
      print(f"  ✅ {text}")


## Quick Usage

# Simple usage
clean_df = preprocess_text_data(
  input_csv="your_data.csv",
  output_csv="clean_data.csv",
  text_column="text"
)

# With custom keywords
custom_keywords = {
  "company_specific": ["competitor_name", "banned_term"],
  "profanity": ["additional_bad_word"]
}

clean_df = preprocess_text_data(
  input_csv="your_data.csv",
  output_csv="clean_data.csv",
  text_column="text",
  custom_keywords=custom_keywords
)
